# Notebook 3: Audio Input with Allosaurus

## What this notebook covers
- Running real audio through a phoneme recogniser (allosaurus), not typing phonemes by hand
- Allosaurus outputs raw IPA phones directly, no G2P needed on the hypothesis side
- Wiring real audio into the PronunciationEvaluator built in Notebook 1
- Where this breaks down (accents, mic quality, short names) and why that matters for a take-home

Built as preparation for the NameCoach take-home evaluation task.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
sys.path.append('../src')

from g2p import g2p
from per import phoneme_error_rate, error_breakdown
from evaluator import PronunciationEvaluator

In [3]:
# Libraries already installed via pip in terminal
from allosaurus.app import read_recognizer

# This downloads/loads the pretrained model on first run, can take a moment
model = read_recognizer()

print("Allosaurus model loaded.")

Allosaurus model loaded.


In [13]:
import soundfile as sf
import os

audio_dir = "audio"

files = {
    "hello": "hello.mp3.mpeg",
    "slow_hello": "hello_slow.mp3.mpeg",
    "slow_hello2": "better_hello.mp3.mpeg",
    "arjun": "arjun.mp3.mpeg",
    "saoirse": "saoirse.mp3.mpeg",
}

wav_paths = {}

for name, fname in files.items():
    src = os.path.join(audio_dir, fname)
    dst = os.path.join(audio_dir, f"{name}.wav")
    try:
        data, samplerate = sf.read(src)
        sf.write(dst, data, samplerate)
        wav_paths[name] = dst
        print(f"{name}: OK, {samplerate} Hz, {len(data)/samplerate:.2f}s -> {dst}")
    except Exception as e:
        print(f"{name}: FAILED, {type(e).__name__}: {e}")

hello: OK, 48000 Hz, 3.10s -> audio\hello.wav
slow_hello: OK, 48000 Hz, 1.70s -> audio\slow_hello.wav
slow_hello2: OK, 48000 Hz, 3.34s -> audio\slow_hello2.wav
arjun: FAILED, LibsndfileError: Unspecified internal error.
saoirse: OK, 48000 Hz, 3.38s -> audio\saoirse.wav


In [5]:
import static_ffmpeg
static_ffmpeg.add_paths()  # fetches ffmpeg/ffprobe binaries on first run, then wires them into PATH

from pydub import AudioSegment

src = "audio/arjun.mp3.mpeg"
dst = "audio/arjun.wav"

# from_file() with no format= lets ffmpeg sniff the real container,
# instead of trusting the (possibly wrong) file extension
audio = AudioSegment.from_file(src)
audio.export(dst, format="wav")

print(f"Converted arjun: {len(audio)/1000:.2f}s, {audio.frame_rate} Hz")

Converted arjun: 2.83s, 48000 Hz


In [14]:
import soundfile as sf

for name in ["hello", "arjun", "saoirse", "slow_hello", "slow_hello2"]:
    path = f"audio/{name}.wav"
    data, sr = sf.read(path)
    print(f"{name}: {sr} Hz, {len(data)/sr:.2f}s, {len(data)} samples")

hello: 48000 Hz, 3.10s, 148608 samples
arjun: 48000 Hz, 2.83s, 135936 samples
saoirse: 48000 Hz, 3.38s, 162432 samples
slow_hello: 48000 Hz, 1.70s, 81792 samples
slow_hello2: 48000 Hz, 3.34s, 160128 samples


In [15]:
from allosaurus.app import read_recognizer

model = read_recognizer()

results = {}
for name in ["hello", "arjun", "saoirse", "slow_hello", "slow_hello2"]:
    path = f"audio/{name}.wav"
    hypothesis = model.recognize(path)
    results[name] = hypothesis.split()
    print(f"{name}: {results[name]}")

hello: ['a', 'l', 'o']
arjun: ['a', 'ɾ', 'd͡ʒ', 'y', 'ʊ', 'uː']
saoirse: ['s', 'œ', 'ɾ', 't͡ʃʲ', 'ɻ̩']
slow_hello: ['ɒ', 'l', 'ɔ', 'uə', 'ɪ']
slow_hello2: ['a', 'l', 'o']


In [12]:
evaluator = PronunciationEvaluator()

for name in ["hello", "arjun", "saoirse", "slow_hello"]:
    result = evaluator.evaluate(word=name, spoken_phonemes=results[name])
    evaluator.print_report(result)

PronunciationEvaluator ready.

Word:           hello
Reference:      ['h', 'ə', 'l', 'oʊ']
  Source:       CMUdict (confidence: HIGH)
Spoken:         ['a', 'l', 'o']
PER:            0.75
Accuracy:       25/100
Grade:          POOR
--------------------------------------------------

Word:           arjun
Reference:      ['ɑː', 'ɹ', 'd', 'ʒ', 'ʌ', 'n']
  Source:       phonemizer (confidence: LOW)
Spoken:         ['a', 'ɾ', 'd', 'ʒ', 'y', 'ʊ', 'uː']
PER:            0.833
Accuracy:       17/100
Grade:          POOR
--------------------------------------------------

Word:           saoirse
Reference:      ['s', 'ɜː', 'ʃ', 'ə']
  Source:       phonemizer (confidence: LOW)
Spoken:         ['s', 'œ', 'ɾ', 't', 'ʃʲ', 'ɻ̩']
PER:            1.25
Accuracy:       0/100
Grade:          POOR
--------------------------------------------------

Word:           slow_hello
Reference:      ['s', 'l', 'o', 'ʊ', ' ', 'h', 'ə', 'l', 'o', 'ʊ']
  Source:       phonemizer (confidence: LOW)
Spoken:         ['ɒ'